# Emerging Tech Lab - Solvent-Screen Opentrons Protocol

## Solvent-screen protocol setup

In [ ]:
from solvent_screen_opentrons_helpers import (
    log_step,
    load_source_plates,
    build_solvent_screen_plate_map,
    validate_volume_for_p300,
    dispense_to_wells_with_tip_changes,
    set_robot_speeds,
    log_absolute_time_tick,
    summarise_solvent_screen_records,
)

import opentrons.execute
protocol = opentrons.execute.get_protocol_api("2.19")

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location=2,
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location=1,
)

# Load one or more 8-well source plates.
# Deck slots must not overlap with the 48-well plate, tip rack, or other labware.
# To add a third source plate, use SOURCE_PLATE_LOCATIONS = [3, 4, 5]
# and then define plate_8_3 = source_plates["plate_8_3"].
SOURCE_PLATE_LOCATIONS = [3, 4]
source_plates = load_source_plates(
    protocol=protocol,
    labware_name="greenaway_8_wellplate_20000ul",
    plate_locations=SOURCE_PLATE_LOCATIONS,
    name_prefix="plate_8",
)
plate_8_1 = source_plates["plate_8_1"]
plate_8_2 = source_plates["plate_8_2"]

# Load pipette
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left",  # change to "right" if needed
    tip_racks=[pip_rack],
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

ASPIRATE_RATE = 70
STANDARD_DISPENSE_RATE = 70
SLOW_DISPENSE_RATE = 10  # used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15
MAX_DISPENSE = 200
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180
TIP_CHANGE_INTERVAL = 3
TRANSFER_MODE = "fast"  # "fast" = one tip per reagent/solvent source; "accurate" = change tips regularly

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
    max_head_speed=400,
)

# -----------------------------
# User setup: source locations and target layout
# -----------------------------

# Source plates hold reagent stocks. The 48-well plate holds reaction conditions.
# 8-well source plate example layout:
# plate_8_1 A1-A2: diamine stocks in different solvents
# plate_8_1 B1-B2: dialdehyde stocks in matching solvents
# plate_8_2 A1/A4: additional diamine stocks
# plate_8_2 B1/B4: additional dialdehyde stocks
# To add more solvent conditions, add entries below and point their source wells
# to whichever loaded source plate contains that stock.
#
# Solvent-screen target layout:
# Each solvent condition is run in triplicate.
# Replicates are arranged vertically within one column block.
#
# Conditions 1-8 use the upper half of the 48-well plate:
#   condition 1 -> A1, B1, C1
#   condition 2 -> A2, B2, C2
#   ...
#   condition 8 -> A8, B8, C8
#
# Conditions 9-16 use the lower half:
#   condition 9  -> D1, E1, F1
#   condition 10 -> D2, E2, F2
#   ...
#   condition 16 -> D8, E8, F8
SOLVENT_CONDITIONS = {
    "chloroform": {
        "diamine_source": plate_8_1["A1"],
        "dialdehyde_source": plate_8_1["B1"],
    },
    "methanol": {
        "diamine_source": plate_8_1["A2"],
        "dialdehyde_source": plate_8_1["B2"],
    },
    "chloroform_methanol_1to1": {
        "diamine_source": plate_8_2["A1"],
        "dialdehyde_source": plate_8_2["B1"],
    },
    "hexane": {
        "diamine_source": plate_8_2["A4"],
        "dialdehyde_source": plate_8_2["B4"],
        # Optional override example:
        # "target_index": 9,  # would force D1, E1, F1 regardless of list order
    },
}

# Select the solvent condition(s) for this run.
# The order of this list controls target-well assignment unless a condition defines target_index.
CONDITIONS_TO_RUN = [
    "chloroform",                  # -> A1, B1, C1
    "methanol",                    # -> A2, B2, C2
    "chloroform_methanol_1to1",    # -> A3, B3, C3
    "hexane",                      # -> A4, B4, C4
]

if len(CONDITIONS_TO_RUN) == 0:
    raise ValueError("Select at least one solvent condition to run.")

# -----------------------------
# User setup: dispense volumes
# -----------------------------

# Enter the calculated volume for each stock solution per reaction vial.
# These volumes are applied to every replicate well in this run.
volume_of_diamine = 1000  # uL per well
volume_of_dialdehyde = 200  # uL per well

# -----------------------------
# Build and validate the plate map
# -----------------------------

validate_volume_for_p300(volume_of_diamine, "Diamine", protocol=protocol)
validate_volume_for_p300(volume_of_dialdehyde, "Dialdehyde", protocol=protocol)

condition_plate_map = build_solvent_screen_plate_map(
    solvent_conditions=SOLVENT_CONDITIONS,
    conditions_to_run=CONDITIONS_TO_RUN,
    protocol=protocol,
)

## Basic OT-2 sanity test

In [ ]:
# -----------------------------
# Basic OT-2 sanity test
# -----------------------------

protocol.home()

log_step(protocol, "Starting basic OT-2 sanity test.")

# Test tip pickup/drop
pip_300.pick_up_tip()
log_step(protocol, "Picked up one tip successfully.")

# Test movement to source and target wells
pip_300.move_to(plate_8_1["A1"].top())
log_step(protocol, "Moved to source plate_8_1 A1 top.")

pip_300.move_to(plate_48["A1"].top())
log_step(protocol, "Moved to target well A1 top.")

pip_300.drop_tip()
log_step(protocol, "Dropped tip successfully.")

protocol.home()
log_step(protocol, "Basic OT-2 sanity test complete.")

## Automated solvent-screen execution

In [ ]:
# -----------------------------
# Automated solvent-screen execution
# Correct addition order:
#   1. diamine solution
#   2. dialdehyde solution, slow/dropwise
# -----------------------------

# Store dispense records for later inspection.
diamine_dispense_records = {}
dialdehyde_dispense_records = {}

# Add diamine solution to all replicate wells for each solvent condition.
# This does not start the imine reaction until dialdehyde is added.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    diamine_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["diamine_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_diamine,
        dispense_rate=STANDARD_DISPENSE_RATE,
        reagent_name=f"diamine stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=False,
        log_absolute_time=False,
    )

# Add dialdehyde solution slowly/dropwise to start the reaction for each solvent condition.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    log_absolute_time_tick(protocol, f"Starting dialdehyde addition for solvent-screen condition {condition_name}.")

    dialdehyde_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["dialdehyde_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_dialdehyde,
        dispense_rate=SLOW_DISPENSE_RATE,
        reagent_name=f"dialdehyde stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=True,
        log_absolute_time=True,
    )

    log_absolute_time_tick(protocol, f"Finished dialdehyde addition for solvent-screen condition {condition_name}.")

# Reset dispense rate and summarise the run.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Solvent-screen summary by condition:")
summarise_solvent_screen_records(
    dispense_records_by_condition=dialdehyde_dispense_records,
    protocol=protocol,
)

protocol.home()